<!-- ### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts. -->

<!-- ### Chatbot Evaluation -->

In [75]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"
from groq import Groq

groq_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

In [76]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['b9b1b5d9-b5c4-4978-9662-a1b2d59e500d',
  '906e18c6-0466-4f1b-9eea-edc825bff832',
  '2621df57-e540-410d-9483-7fda29b950c1',
  'f498180b-b8c4-48bf-9609-e53cc8caed13',
  '410316b3-075f-4051-8978-4de8fb7c5f19'],
 'count': 5,
 'as_of': '2026-06-08T12:15:54.405253726Z'}

<!-- ### Define Metrics (LLM As A Judge) -->


In [77]:
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response = groq_client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      temperature=0,
      messages=[
            {"role":"system","content":eval_instructions},
            {"role":"user","content":user_content}
      ]
      ).choices[0].message.content

      return response == "CORRECT"

In [78]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

<!-- ### Run Evaluations -->

In [79]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(
    question: str,
    model: str = "llama-3.3-70b-versatile",
    instructions: str = default_instructions,
) -> str:

    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [80]:
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="llama-3.3-70b-versatile"
        )
    }

In [81]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4o-mini-chatbot"
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-71cbcbab' at:
https://smith.langchain.com/o/3dfae94d-f82a-449c-b0d0-ee84299e6659/datasets/a09e8f00-1d8f-49f9-af83-b7107df238b2/compare?selectedSessions=547f594d-7940-44e0-8aae-fb998a07ab79




5it [00:03,  1.36it/s]


In [82]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="llama-3.3-70b-versatile")}

In [83]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-4-turbo-chatbot-873ba700' at:
https://smith.langchain.com/o/3dfae94d-f82a-449c-b0d0-ee84299e6659/datasets/a09e8f00-1d8f-49f9-af83-b7107df238b2/compare?selectedSessions=40137383-c034-4b83-8643-1afc64c98da4




5it [00:03,  1.51it/s]
